# Thanglish Agent — RunPod RTX 5090

This notebook controls the full project by running the scripts in this folder. **The actual 18.8 GB pretrained model, 903-file dataset and original tokenizer must be provided separately.**

Model ID: `1QqPYEYoC-AQDYJWRjVKCyDgjtw9VNAga`. The original Drive link may be quota-blocked; no URL variant can bypass a source quota.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT=Path("/workspace/thanglish_transliteration")
assert (ROOT/"scripts"/"download_drive.py").exists(), "Unzip project into /workspace/thanglish_transliteration"
print("Project:",ROOT)


## Cell 1 — Python dependencies and GPU check

In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-r",str(ROOT/"requirements.txt")],check=True)
subprocess.run([sys.executable,str(ROOT/"scripts"/"check_environment.py")],check=True)


## Cell 2 — Model download (optional)

Set `RUN_DOWNLOAD=True` only when you want to download. HTML/failed quota responses are rejected and not promoted to model files.

In [ ]:
MODEL_ID="1QqPYEYoC-AQDYJWRjVKCyDgjtw9VNAga"
RUN_DOWNLOAD=False
DOWNLOAD_METHOD="gdown"  # gdown or direct
if RUN_DOWNLOAD:
    subprocess.run([sys.executable,str(ROOT/"scripts"/"download_drive.py"),"--asset","model","--method",DOWNLOAD_METHOD,"--id",MODEL_ID],check=True)
else:
    print("Skipped download. Current checkpoint:", (ROOT/"model_download"/"tl_clm_model.zip").exists())


## Cell 3 — Dataset download (optional)

The 903-file dataset ID was **not supplied**. Set it to your actual Drive ID before enabling download.

In [ ]:
DATASET_ID=""  # Supply the real Google Drive dataset ZIP ID
RUN_DATASET_DOWNLOAD=False
if RUN_DATASET_DOWNLOAD:
    assert DATASET_ID,"Provide actual dataset ID"
    subprocess.run([sys.executable,str(ROOT/"scripts"/"download_drive.py"),"--asset","dataset","--id",DATASET_ID],check=True)
else:
    print("Skipped dataset download")


## Cell 4 — Import training data

By default uses 60 **example pairs** only, not the real 903-file dataset. Switch off demo once the real dataset ZIP is available.

In [ ]:
DEMO_ONLY=True
if DEMO_ONLY:
    subprocess.run([sys.executable,str(ROOT/"scripts"/"prepare_dataset.py"),"--demo"],check=True)
else:
    zip_path=ROOT/"data"/"raw"/"thanglish_data.zip"
    cmd=[sys.executable,str(ROOT/"scripts"/"prepare_dataset.py")]
    if zip_path.exists():cmd += ["--zip",str(zip_path)]
    subprocess.run(cmd,check=True)


## Cell 5 — 60 reference transliteration samples

In [ ]:
subprocess.run([sys.executable,str(ROOT/"scripts"/"evaluate_rules.py")],check=True)


## Cell 6 — Run automated tests

In [ ]:
subprocess.run([sys.executable,"-m","pytest","-q",str(ROOT/"tests")],check=True,cwd=str(ROOT))


## Cell 7 — Exact parameters (requires clean real checkpoint)

In [ ]:
CHECKPOINT=ROOT/"model_download"/"tl_clm_model.zip"
if CHECKPOINT.is_file():
    subprocess.run([sys.executable,str(ROOT/"scripts"/"parameter_report.py")],check=True)
else:
    print("Missing checkpoint; exact parameter count cannot yet be measured")


## Cell 8 — Deterministic Tamil → Thanglish baseline

In [ ]:
TEXT="தமிழ் மொழியின் வரலாறு மிகவும் பழமையானது"
subprocess.run([sys.executable,str(ROOT/"scripts"/"infer.py"),"--mode","rules","--text",TEXT],check=True)


## Cell 9 — Neural model inference (requires checkpoint + original tokenizer)

In [ ]:
TOKENIZER=ROOT/"models"/"tamil_sp_v4.model"
if CHECKPOINT.is_file() and TOKENIZER.is_file():
    subprocess.run([sys.executable,str(ROOT/"scripts"/"infer.py"),"--mode","model","--text",TEXT],check=True)
else:
    print("Skipping model inference: real checkpoint/tokenizer not provided")


## Cell 10 — OPTIONAL LoRA training

Do **not** enable until you have verified the real checkpoint, tokenizer, train/val pairs and task direction. This trains adapters only, not the pretrained CLM from scratch.

In [ ]:
RUN_LORA_TRAINING=False
if RUN_LORA_TRAINING:
    assert not DEMO_ONLY, "Do not use the small demo set for your final model"
    subprocess.run([sys.executable,str(ROOT/"scripts"/"train_lora.py")],check=True)
else:
    print("Optional LoRA training skipped")
